In [1]:
# Customer Churn Prediction - Random Forest
# This script trains a Random Forest model using historical
# telecom customer data and predicts churn for new customers.



In [2]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder



In [3]:
# Load historical churn data

file_path = "../data/Prediction_Data.xlsx"

data = pd.read_excel(
    file_path,
    sheet_name="vw_ChurnData"
)

print("Dataset Shape:", data.shape)
print(data.head())



FileNotFoundError: [Errno 2] No such file or directory: '../data/Prediction_Data.xlsx'

In [ ]:
# Data preprocessing

# Remove columns that are not used for prediction

data = data.drop(
    ["Customer_ID", "Churn_Category", "Churn_Reason"],
    axis=1
)



In [ ]:
# Encode categorical variables

columns_to_encode = [
    "Gender",
    "Married",
    "State",
    "Value_Deal",
    "Phone_Service",
    "Multiple_Lines",
    "Internet_Service",
    "Internet_Type",
    "Online_Security",
    "Online_Backup",
    "Device_Protection_Plan",
    "Premium_Support",
    "Streaming_TV",
    "Streaming_Movies",
    "Streaming_Music",
    "Unlimited_Data",
    "Contract",
    "Paperless_Billing",
    "Payment_Method"
]

label_encoders = {}

for column in columns_to_encode:
    encoder = LabelEncoder()
    data[column] = encoder.fit_transform(data[column])
    label_encoders[column] = encoder



In [ ]:
# Encode target variable

# Stayed = 0
# Churned = 1

data["Customer_Status"] = data["Customer_Status"].map({
    "Stayed": 0,
    "Churned": 1
})

print(data["Customer_Status"].value_counts())



In [ ]:
# Define features and target

X = data.drop(
    "Customer_Status",
    axis=1
)

y = data["Customer_Status"]



In [ ]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)



In [ ]:
# Train Random Forest model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

print("Model training completed.")



In [ ]:
# Generate predictions on test data

y_pred = rf_model.predict(X_test)



In [ ]:
# Model evaluation

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Stayed", "Churned"]
    )
)



In [ ]:
# Feature importance

importances = rf_model.feature_importances_

indices = np.argsort(importances)[::-1]



In [ ]:
# Visualize feature importance

plt.figure(figsize=(15, 6))

sns.barplot(
    x=importances[indices],
    y=X.columns[indices]
)

plt.title("Feature Importance")
plt.xlabel("Relative Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()



In [ ]:
# Save trained model and encoders

joblib.dump(
    rf_model,
    "../models/random_forest_churn_model.pkl"
)

joblib.dump(
    label_encoders,
    "../models/churn_label_encoders.pkl"
)

print("Model and encoders saved successfully.")



In [ ]:
# Load new customer data

new_data = pd.read_excel(
    file_path,
    sheet_name="vw_JoinData"
)

print("New Customer Data Shape:", new_data.shape)

print(new_data.head())



In [ ]:
# Preserve original customer data

original_data = new_data.copy()



In [ ]:
# Prepare new data for prediction

new_data = new_data.drop(
    [
        "Customer_ID",
        "Customer_Status",
        "Churn_Category",
        "Churn_Reason"
    ],
    axis=1
)



In [ ]:
# Encode new customer data

# Use the encoders created during model training

for column in new_data.select_dtypes(
    include=["object"]
).columns:

    new_data[column] = label_encoders[column].transform(
        new_data[column]
    )



In [ ]:
# Generate churn predictions

new_predictions = rf_model.predict(
    new_data
)



In [ ]:
# Add predictions to original data

original_data["Customer_Status_Predicted"] = new_predictions

original_data["Customer_Status_Predicted_Label"] = (
    original_data["Customer_Status_Predicted"]
    .map({
        0: "Stayed",
        1: "Churned"
    })
)



In [ ]:
# Filter predicted churners

predicted_churners = original_data[
    original_data["Customer_Status_Predicted"] == 1
].copy()

print(
    f"Predicted Churners: {len(predicted_churners)}"
)

display(predicted_churners.head())



In [ ]:
# Export predictions

output_path = "../data/Predictions.csv"

predicted_churners.to_csv(
    output_path,
    index=False
)

print(
    f"Predictions saved to: {output_path}"
)



In [ ]:
# Final summary

print("==========================================")
print("CHURN PREDICTION COMPLETED")
print("==========================================")

print(
    f"Customers evaluated: {len(original_data)}"
)

print(
    f"Predicted churners: {len(predicted_churners)}"
)

print(
    f"Output file: {output_path}"
)
